In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from faker import Faker

# Initialize faker for realistic data
fake = Faker()

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate customer data (simulating Doha-based customers)
customers = []
customer_ids = []
for i in range(25):  # Generate 25 customers
    customer_id = f"CUST-{1000 + i}"
    customer_ids.append(customer_id)
    
    # Generate some Arabic names for realism (transliterated)
    arabic_names = [
        "Ahmed Al-Mansoori", "Fatima Al-Kuwari", "Mohammed Al-Thani", "Aisha Al-Attiyah",
        "Khalid Al-Sulaiti", "Noor Al-Mohannadi", "Omar Al-Jaber", "Layla Al-Emadi",
        "Hassan Al-Maadeed", "Sara Al-Henzab", "Ali Al-Baker", "Mariam Al-Khalaf",
        "Yousef Al-Ansari", "Huda Al-Rumaihi", "Abdullah Al-Mohammed"
    ]
    
    # Mix with international names for Qatar Airways context
    international_names = [
        "James Wilson", "Sophie Martin", "Robert Chen", "Elena Rodriguez",
        "David Kim", "Anna Schmidt", "Michael Brown", "Lisa Taylor"
    ]
    
    all_names = arabic_names + international_names
    customer_name = random.choice(all_names)
    
    # Generate company names (mix of Qatari and international)
    company_types = ["Trading", "Travel", "Corporate", "Logistics", "Services", "Aviation"]
    company_prefix = random.choice(["Al-", "Qatar ", "Gulf ", "Middle East ", "International "])
    company_suffix = random.choice(["W.L.L.", "LLC", "Group", "Company", "Est."])
    company_name = f"{company_prefix}{random.choice(company_types)} {company_suffix}"
    
    # Generate contact details
    phone = f"+974 {random.randint(3000, 7999)} {random.randint(1000, 9999)}"
    email = f"accounts@{company_name.replace(' ', '').replace('.', '').replace('-', '').lower()}.com"
    
    customers.append({
        'Customer ID': customer_id,
        'Customer Name': customer_name,
        'Company Name': company_name,
        'Contact Person': customer_name,
        'Phone': phone,
        'Email': email,
        'Credit Limit (QAR)': random.choice([50000, 100000, 150000, 200000, 250000]),
        'Payment Terms': random.choice(["Net 30", "Net 15", "Net 45", "Due on receipt"])
    })

# Create customer dataframe
customer_df = pd.DataFrame(customers)

# Generate invoice data
invoices = []
invoice_counter = 1000

# Define base date for aging
base_date = datetime(2025, 1, 15)

for customer_id in customer_ids:
    # Generate 3-8 invoices per customer
    num_invoices = random.randint(3, 8)
    
    for _ in range(num_invoices):
        invoice_counter += 1
        invoice_date = base_date - timedelta(days=random.randint(0, 120))
        
        # Get payment terms and calculate due date
        payment_terms = customer_df[customer_df['Customer ID'] == customer_id]['Payment Terms'].values[0]
        
        if payment_terms == "Due on receipt":
            # For "Due on receipt", due date is same as invoice date
            due_date = invoice_date
        else:
            # Extract number from "Net XX"
            try:
                days = int(payment_terms.split()[1])
                due_date = invoice_date + timedelta(days=days)
            except (ValueError, IndexError):
                # Default to 30 days if there's an issue
                due_date = invoice_date + timedelta(days=30)
        
        # Create aging categories
        days_overdue = (base_date - due_date).days
        
        if days_overdue <= 0:
            aging_bucket = "Current (0-30)"
        elif 1 <= days_overdue <= 30:
            aging_bucket = "Current (0-30)"
        elif 31 <= days_overdue <= 60:
            aging_bucket = "31-60 days"
        elif 61 <= days_overdue <= 90:
            aging_bucket = "61-90 days"
        else:
            aging_bucket = "90+ days"
        
        # Generate invoice amount
        amount = round(random.uniform(5000, 50000), 2)
        
        # Some invoices are partially paid
        if random.random() > 0.3:  # 70% chance of having some payment
            paid_amount = round(amount * random.uniform(0, 1), 2)
            if random.random() > 0.8:  # 20% chance of being fully paid
                paid_amount = amount
        else:
            paid_amount = 0
        
        outstanding = amount - paid_amount
        
        invoices.append({
            'Invoice Number': f"INV-{invoice_counter}",
            'Customer ID': customer_id,
            'Invoice Date': invoice_date.date(),
            'Due Date': due_date.date(),
            'Invoice Amount (QAR)': amount,
            'Paid Amount (QAR)': paid_amount,
            'Outstanding Amount (QAR)': outstanding,
            'Aging Bucket': aging_bucket,
            'Days Overdue': max(0, days_overdue),
            'Status': 'Paid' if outstanding == 0 else 'Partial' if paid_amount > 0 else 'Unpaid',
            'Product/Service': random.choice([
                "Flight Tickets", "Cargo Service", "VIP Lounge", "Baggage Fees",
                "Charter Service", "Aviation Fuel", "Maintenance Service", "Catering Service"
            ]),
            'LPO Number': f"LPO-{random.randint(5000, 5999)}" if random.random() > 0.2 else "",  # Some invoices without LPO
            'Payment Terms': payment_terms
        })

# Create invoices dataframe
invoices_df = pd.DataFrame(invoices)

# Generate transaction/payment history
transactions = []
transaction_id = 5000

for index, invoice in invoices_df.iterrows():
    if invoice['Paid Amount (QAR)'] > 0:
        # Create multiple payments for partially paid invoices
        if invoice['Status'] == 'Partial' and random.random() > 0.5:
            num_payments = random.randint(2, 3)
            payment_split = invoice['Paid Amount (QAR)'] / num_payments
        else:
            num_payments = 1
            payment_split = invoice['Paid Amount (QAR)']
        
        for i in range(num_payments):
            transaction_id += 1
            payment_date = invoice['Due Date'] - timedelta(days=random.randint(-10, 20))
            
            # Ensure payment date is not before invoice date
            if payment_date < invoice['Invoice Date']:
                payment_date = invoice['Invoice Date'] + timedelta(days=random.randint(1, 30))
            
            transactions.append({
                'Transaction ID': f"TRN-{transaction_id}",
                'Invoice Number': invoice['Invoice Number'],
                'Customer ID': invoice['Customer ID'],
                'Payment Date': payment_date,
                'Payment Amount (QAR)': round(payment_split, 2) if num_payments > 1 else payment_split,
                'Payment Method': random.choice(["Bank Transfer", "Cheque", "Cash", "Credit Card"]),
                'Reference Number': f"REF-{random.randint(10000, 99999)}",
                'Bank': random.choice(["QNB", "CBQ", "QIB", "HSBC", "Doha Bank"]),
                'Status': random.choice(["Cleared", "Pending Clearance", "Returned"]) if random.random() > 0.8 else "Cleared"
            })

transactions_df = pd.DataFrame(transactions) if transactions else pd.DataFrame(columns=['Transaction ID', 'Invoice Number', 'Customer ID', 'Payment Date', 'Payment Amount (QAR)', 'Payment Method', 'Reference Number', 'Bank', 'Status'])

# Generate customer statement data
statements = []
for customer_id in customer_ids:
    customer_invoices = invoices_df[invoices_df['Customer ID'] == customer_id]
    total_outstanding = customer_invoices['Outstanding Amount (QAR)'].sum()
    
    if total_outstanding > 0 or random.random() > 0.3:  # Include some customers with zero balance
        current_balance = customer_invoices[customer_invoices['Aging Bucket'] == 'Current (0-30)']['Outstanding Amount (QAR)'].sum()
        bucket_31_60 = customer_invoices[customer_invoices['Aging Bucket'] == '31-60 days']['Outstanding Amount (QAR)'].sum()
        bucket_61_90 = customer_invoices[customer_invoices['Aging Bucket'] == '61-90 days']['Outstanding Amount (QAR)'].sum()
        bucket_90_plus = customer_invoices[customer_invoices['Aging Bucket'] == '90+ days']['Outstanding Amount (QAR)'].sum()
        
        statements.append({
            'Customer ID': customer_id,
            'Statement Date': base_date.date(),
            'Total Outstanding (QAR)': total_outstanding,
            'Current Balance (QAR)': current_balance,
            '31-60 Days (QAR)': bucket_31_60,
            '61-90 Days (QAR)': bucket_61_90,
            '90+ Days (QAR)': bucket_90_plus,
            'Last Payment Date': transactions_df[transactions_df['Customer ID'] == customer_id]['Payment Date'].max() if not transactions_df.empty and customer_id in transactions_df['Customer ID'].values else None,
            'Last Payment Amount': transactions_df[transactions_df['Customer ID'] == customer_id]['Payment Amount (QAR)'].max() if not transactions_df.empty and customer_id in transactions_df['Customer ID'].values else 0,
            'Follow-up Required': 'Yes' if total_outstanding > 50000 or (customer_invoices['Days Overdue'].max() > 60 and total_outstanding > 10000) else 'No',
            'Priority': 'High' if bucket_90_plus > 20000 or total_outstanding > 75000 else 'Medium' if total_outstanding > 30000 else 'Low'
        })

statements_df = pd.DataFrame(statements)

# Generate collection follow-up log
followups = []
followup_id = 100

for index, statement in statements_df.iterrows():
    if statement['Follow-up Required'] == 'Yes':
        customer_id = statement['Customer ID']
        customer_name = customer_df[customer_df['Customer ID'] == customer_id]['Customer Name'].values[0]
        company_name = customer_df[customer_df['Customer ID'] == customer_id]['Company Name'].values[0]
        
        num_followups = random.randint(1, 4)
        for i in range(num_followups):
            followup_id += 1
            followup_date = base_date - timedelta(days=random.randint(1, 30))
            
            # Different follow-up reasons based on aging
            if statement['90+ Days (QAR)'] > 0:
                discussion_options = [
                    "Escalated to legal department",
                    "Final demand notice sent",
                    "Payment plan negotiated",
                    "Legal action being considered"
                ]
            elif statement['61-90 Days (QAR)'] > 0:
                discussion_options = [
                    "Promised payment by end of month",
                    "Requested copy of all invoices",
                    "Disputed amount - need clarification",
                    "Payment processing delayed"
                ]
            else:
                discussion_options = [
                    "Reminder sent for upcoming payment",
                    "Confirmed receipt of invoice",
                    "Awaiting approval from finance manager",
                    "Bank transfer in process"
                ]
            
            followups.append({
                'Follow-up ID': f"FUP-{followup_id}",
                'Customer ID': customer_id,
                'Customer Name': customer_name,
                'Company Name': company_name,
                'Follow-up Date': followup_date.date(),
                'Contact Method': random.choice(["Phone Call", "Email", "Official Letter", "Visit"]),
                'Contact Person': random.choice(["Accounts Manager", "Finance Director", "Owner", "Procurement Manager"]),
                'Discussion Summary': random.choice(discussion_options),
                'Next Action': random.choice([
                    "Follow up in 3 days",
                    "Send statement copy",
                    "Schedule meeting with manager",
                    "Escalate to senior management",
                    "Wait for bank confirmation",
                    "Send reminder email"
                ]),
                'Responsible Person': random.choice(["AR Specialist", "Finance Assistant", "Collection Officer"]),
                'Status': random.choice(["Completed", "Pending", "In Progress", "Escalated"]),
                'Next Follow-up Date': (followup_date + timedelta(days=random.randint(2, 7))).date()
            })

followups_df = pd.DataFrame(followups) if followups else pd.DataFrame(columns=['Follow-up ID', 'Customer ID', 'Customer Name', 'Company Name', 'Follow-up Date', 'Contact Method', 'Contact Person', 'Discussion Summary', 'Next Action', 'Responsible Person', 'Status', 'Next Follow-up Date'])

# Create Excel writer object
with pd.ExcelWriter('aged_accounts_receivable_project.xlsx', engine='openpyxl') as writer:
    # Write customer master data
    customer_df.to_excel(writer, sheet_name='Customer Master', index=False)
    
    # Write invoice data
    invoices_df.to_excel(writer, sheet_name='Invoice Details', index=False)
    
    # Write aged debtors summary
    aged_summary = invoices_df.groupby(['Customer ID', 'Aging Bucket'])['Outstanding Amount (QAR)'].sum().unstack(fill_value=0)
    aged_summary['Total Outstanding'] = aged_summary.sum(axis=1)
    aged_summary = aged_summary.reset_index()
    
    # Merge with customer names
    aged_summary = pd.merge(aged_summary, customer_df[['Customer ID', 'Customer Name', 'Company Name']], on='Customer ID', how='left')
    
    # Reorder columns
    cols = ['Customer ID', 'Customer Name', 'Company Name', 'Total Outstanding'] + [col for col in aged_summary.columns if col not in ['Customer ID', 'Customer Name', 'Company Name', 'Total Outstanding']]
    cols = [col for col in cols if col in aged_summary.columns]  # Filter columns that exist
    aged_summary = aged_summary[cols]
    
    aged_summary.to_excel(writer, sheet_name='Aged Debtors Summary', index=False)
    
    # Write customer statements
    statements_df.to_excel(writer, sheet_name='Customer Statements', index=False)
    
    # Write transaction history
    transactions_df.to_excel(writer, sheet_name='Payment History', index=False)
    
    # Write collection follow-up log
    followups_df.to_excel(writer, sheet_name='Collection Follow-up', index=False)
    
    # Create dashboard summary
    if not invoices_df.empty:
        total_outstanding = invoices_df['Outstanding Amount (QAR)'].sum()
        total_invoices = invoices_df['Invoice Amount (QAR)'].sum()
        
        summary_data = {
            'Metric': [
                'Total Customers',
                'Active AR Customers',
                'Total Invoices Outstanding',
                'Total Outstanding Amount (QAR)',
                'Average Days Overdue',
                'Customers > 60 days overdue',
                'Collection Rate (Last 30 days)',
                'Largest Single Invoice Outstanding',
                'Customers at Credit Limit',
                'Total Overdue (>30 days)',
                'Aging % Current',
                'Aging % 31-60 days',
                'Aging % 61-90 days',
                'Aging % 90+ days'
            ],
            'Value': [
                len(customer_df),
                invoices_df['Customer ID'].nunique(),
                len(invoices_df[invoices_df['Outstanding Amount (QAR)'] > 0]),
                round(total_outstanding, 2),
                round(invoices_df['Days Overdue'].mean(), 1) if not invoices_df.empty else 0,
                len(invoices_df[invoices_df['Days Overdue'] > 60]['Customer ID'].unique()) if not invoices_df.empty else 0,
                f"{round((transactions_df['Payment Amount (QAR)'].sum() / total_invoices) * 100, 1) if total_invoices > 0 else '0.0'}%",
                round(invoices_df['Outstanding Amount (QAR)'].max(), 2) if not invoices_df.empty else 0,
                len([cust for cust in customer_ids if invoices_df[invoices_df['Customer ID'] == cust]['Outstanding Amount (QAR)'].sum() > customer_df[customer_df['Customer ID'] == cust]['Credit Limit (QAR)'].values[0]]) if not invoices_df.empty else 0,
                round(invoices_df[invoices_df['Days Overdue'] > 30]['Outstanding Amount (QAR)'].sum(), 2) if not invoices_df.empty else 0,
                f"{round((invoices_df[invoices_df['Aging Bucket'] == 'Current (0-30)']['Outstanding Amount (QAR)'].sum() / total_outstanding) * 100, 1) if total_outstanding > 0 else '0.0'}%",
                f"{round((invoices_df[invoices_df['Aging Bucket'] == '31-60 days']['Outstanding Amount (QAR)'].sum() / total_outstanding) * 100, 1) if total_outstanding > 0 else '0.0'}%",
                f"{round((invoices_df[invoices_df['Aging Bucket'] == '61-90 days']['Outstanding Amount (QAR)'].sum() / total_outstanding) * 100, 1) if total_outstanding > 0 else '0.0'}%",
                f"{round((invoices_df[invoices_df['Aging Bucket'] == '90+ days']['Outstanding Amount (QAR)'].sum() / total_outstanding) * 100, 1) if total_outstanding > 0 else '0.0'}%"
            ]
        }
    else:
        summary_data = {'Metric': ['No Data Available'], 'Value': ['']}
    
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(writer, sheet_name='Dashboard Summary', index=False)
    
    # Add a reconciliation sheet
    reconciliation_data = []
    for customer_id in customer_ids:
        customer_invoices = invoices_df[invoices_df['Customer ID'] == customer_id]
        customer_payments = transactions_df[transactions_df['Customer ID'] == customer_id]
        
        if not customer_invoices.empty:
            reconciliation_data.append({
                'Customer ID': customer_id,
                'Customer Name': customer_df[customer_df['Customer ID'] == customer_id]['Customer Name'].values[0],
                'Total Invoiced': customer_invoices['Invoice Amount (QAR)'].sum(),
                'Total Payments': customer_payments['Payment Amount (QAR)'].sum() if not customer_payments.empty else 0,
                'Outstanding Balance': customer_invoices['Outstanding Amount (QAR)'].sum(),
                'Difference': customer_invoices['Invoice Amount (QAR)'].sum() - (customer_payments['Payment Amount (QAR)'].sum() if not customer_payments.empty else 0) - customer_invoices['Outstanding Amount (QAR)'].sum(),
                'Reconciled': 'Yes' if abs(customer_invoices['Invoice Amount (QAR)'].sum() - (customer_payments['Payment Amount (QAR)'].sum() if not customer_payments.empty else 0) - customer_invoices['Outstanding Amount (QAR)'].sum()) < 0.01 else 'No'
            })
    
    reconciliation_df = pd.DataFrame(reconciliation_data)
    reconciliation_df.to_excel(writer, sheet_name='Reconciliation', index=False)

print("Dataset created successfully!")
print(f"File saved as: aged_accounts_receivable_project.xlsx")
print("\nExcel File Contents:")
print("1. Customer Master - Customer details and credit limits")
print("2. Invoice Details - All invoices with aging information")
print("3. Aged Debtors Summary - Outstanding amounts by aging bucket")
print("4. Customer Statements - Summary for each customer")
print("5. Payment History - All payment transactions")
print("6. Collection Follow-up - Collection activity log")
print("7. Dashboard Summary - Key metrics overview")
print("8. Reconciliation - AR reconciliation status")

# Print sample data for verification
print("\n" + "="*80)
print("SAMPLE DATA FOR VERIFICATION")
print("="*80)

print("\n1. Customer Master (first 5 rows):")
print(customer_df.head().to_string())

print("\n2. Invoice Details (first 5 rows):")
print(invoices_df[['Invoice Number', 'Customer ID', 'Due Date', 'Outstanding Amount (QAR)', 'Aging Bucket', 'Days Overdue', 'Status']].head().to_string())

print("\n3. Aged Debtors Summary (first 5 rows):")
print(aged_summary.head().to_string())

print("\n4. Key Statistics:")
print(f"Total Outstanding: QAR {invoices_df['Outstanding Amount (QAR)'].sum():,.2f}")
print(f"Number of Invoices: {len(invoices_df)}")
print(f"Overdue Invoices (>30 days): {len(invoices_df[invoices_df['Days Overdue'] > 30])}")
print(f"Customers requiring follow-up: {len(statements_df[statements_df['Follow-up Required'] == 'Yes'])}")






Dataset created successfully!
File saved as: aged_accounts_receivable_project.xlsx

Excel File Contents:
1. Customer Master - Customer details and credit limits
2. Invoice Details - All invoices with aging information
3. Aged Debtors Summary - Outstanding amounts by aging bucket
4. Customer Statements - Summary for each customer
5. Payment History - All payment transactions
6. Collection Follow-up - Collection activity log
7. Dashboard Summary - Key metrics overview
8. Reconciliation - AR reconciliation status

SAMPLE DATA FOR VERIFICATION

1. Customer Master (first 5 rows):
  Customer ID      Customer Name                   Company Name     Contact Person           Phone                                  Email  Credit Limit (QAR)   Payment Terms
0   CUST-1000       Anna Schmidt             Al-Aviation W.L.L.       Anna Schmidt  +974 5253 5012             accounts@alaviationwll.com              100000          Net 15
1   CUST-1001   Aisha Al-Attiyah  International Services W.L.L.   Aish